In [ ]:
'''
# Basic packages
conda install numpy pandas scikit-learn pillow matplotlib jupyter seaborn tqdm -y

# Polars (fast DataFrame lib)
pip install torch torchvision polars torch-geometric pyarrow opencv-python torch_geometric_temporal

# imageio imageio-ffmpeg ultralytics
'''

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import StandardScaler
import math
from pathlib import Path
from collections import defaultdict
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
# import cv2
from glob import glob
import shutil
from tqdm import tqdm

# from ultralytics import YOLO

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, f1_score

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.data import InMemoryDataset

from torch.autograd import Variable

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

# Data


In [ ]:
import polars as pl
import numpy as np

colnum_names = [
    'subject',
    'timestamp',
    'frame',
    ] + [
        f'{point}_{postion}'
        for point in range(17)
        for postion in ['x', 'y', 'z']
        ] + [
            f'label_{label}'
            for label in range(12)
            ]


num_subject = 5
num_timestamp = 20
num_frame = 30

meta_data = [
    (s, t, f)
    for s in range(num_subject)
    for t in range(num_timestamp)
    for f in range(num_frame)
]

meta_array = np.array(meta_data)
num_rows = len(meta_array)
positions_array = np.random.randn(num_rows, 51).round(4)
label_array = np.random.randint(2, size=(num_rows, 12))
full_array = np.hstack([meta_array, positions_array, label_array]) #label_array
df = pl.DataFrame(full_array, schema=colnum_names)

df = df.with_columns([
    pl.col("subject").cast(pl.Int64),
    pl.col("timestamp").cast(pl.Int64),
    pl.col("frame").cast(pl.Int64),
])
df

In [ ]:
df = df.to_pandas()

In [ ]:
df['subject_timestamp_pair'] = list(zip(df['subject'], df['timestamp']))
unique_pairs = df['subject_timestamp_pair'].unique()

In [ ]:
unique_pairs

In [ ]:
train_pairs, test_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42)
train_pairs_list = list(train_pairs)
test_pairs_list = list(test_pairs)

In [ ]:
train_df = df[df['subject_timestamp_pair'].isin(train_pairs_list)].copy()
test_df = df[df['subject_timestamp_pair'].isin(test_pairs_list)].copy()
train_df = train_df.drop(columns=['subject_timestamp_pair'])
test_df = test_df.drop(columns=['subject_timestamp_pair'])
train_df

In [ ]:
train_df.head(20)

In [ ]:
train_df

# Real data

### Train_df

In [ ]:
label_df = pd.read_parquet('/project/ai901504-ai0004/501641_Big/week7/pose_estimate.parquet')

In [ ]:
label_df = label_df.rename(columns={'subject': 'Filename'})
label_df = label_df.drop(columns=[
       'pos_13_x', 'pos_13_y', 'pos_14_x', 'pos_14_y', 'pos_15_x', 'pos_15_y',
       'pos_16_x', 'pos_16_y']) #'pos_11_x', 'pos_11_y', 'pos_12_x', 'pos_12_y', 
label_df.head(3)

In [ ]:
train_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week7/train_pdf_concat.csv")

In [ ]:
train_df.head(3)

In [ ]:
train_df['subject_timestamp_pair'] = list(zip(train_df['Filename'], train_df['Time']))
unique_pairs = train_df['subject_timestamp_pair'].unique()

In [ ]:
def time_str_to_int(tstr):
    """Convert 'tMMSS' to total seconds"""
    minutes = int(tstr[1:3])
    seconds = int(tstr[3:])
    return minutes * 60 + seconds

def time_int_to_str(total_seconds):
    """Convert total seconds to 'tMMSS' format"""
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"t{minutes:02d}{seconds:02d}"

final_df = []

for filename, target_time_str in unique_pairs:
    target_time = time_str_to_int(target_time_str)

    # Get the candidate time windows: t-1, t, t+1
    time_window = [time_int_to_str(target_time + offset) for offset in [-2, -1, 0]]

    # Filter label rows that match filename and fall in the 3-time window
    label_window_rows = label_df[
        (label_df['Filename'] == filename) &
        (label_df['Time'].isin(time_window))
    ].copy()

    feature_row = train_df[
        (train_df['Filename'] == filename) &
        (train_df['Time'] == target_time_str)
    ]

    feature_row = feature_row.drop(columns=['Time'], errors='ignore')  # avoid duplicate column when merging

    # Repeat feature_row to match the number of label rows
    try:
        feature_data = pd.concat([feature_row.iloc[0:1]]*len(label_window_rows), ignore_index=True)
        feature_data.index = label_window_rows.index 
    except:
        print(filename)
        print(target_time_str) # align index so columns add properly
        continue

    merged = pd.concat([label_window_rows, feature_data.drop(columns=['Filename'])], axis=1)

    final_df.append(merged)

# Combine all into one DataFrame
final_df = pd.concat(final_df, ignore_index=True)

In [ ]:
row = train_df[
    (train_df['Filename'] == 'Gary Oldman Interview 4 (Increasing Pressure vs Decreasing Pressure)') &
    (train_df['Time'] == 't0211')
]
print(row)


In [ ]:
final_df

In [ ]:
final_df.isnull().sum()

In [ ]:
final_df.columns

In [ ]:
print(final_df['Filename'].unique())
final_df = final_df[final_df['Filename'] != 'Tom Cruise_s Heated Interview With Matt Lauer Archives TODAY']
print(final_df['Filename'].unique())

### Test_df

In [ ]:
label_test_df = pd.read_parquet('/project/ai901504-ai0004/501641_Big/week7/pose_estimate_test.parquet')
label_test_df = label_test_df.rename(columns={'subject': 'Filename'})
label_test_df = label_test_df.drop(columns=[
       'pos_13_x', 'pos_13_y', 'pos_14_x', 'pos_14_y', 'pos_15_x', 'pos_15_y',
       'pos_16_x', 'pos_16_y'])
label_test_df.head(3)

In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week7/test_submission.csv")

In [ ]:
test_df['subject_timestamp_pair'] = list(zip(test_df['Filename'], test_df['Time']))
unique_pairs = test_df['subject_timestamp_pair'].unique()

In [ ]:
final_test_df = []

for filename, target_time_str in unique_pairs:
    target_time = time_str_to_int(target_time_str)

    # Get the candidate time windows: t-1, t, t+1
    time_window = [time_int_to_str(target_time + offset) for offset in [-2, -1, 0]]

    # Filter label rows that match filename and fall in the 3-time window
    label_window_rows = label_test_df[
        (label_test_df['Filename'] == filename) &
        (label_test_df['Time'].isin(time_window))
    ].copy()

    label_window_rows['subject_timestamp_pair'] = [(filename, target_time_str)] * len(label_window_rows)

    final_test_df.append(label_window_rows)

# Combine all into one DataFrame
final_test_df = pd.concat(final_test_df, ignore_index=True)

In [ ]:
final_test_df

In [ ]:
final_test_df.to_parquet('/project/ai901504-ai0004/501641_Big/week7/jeang_tester.parquet')

In [ ]:
def get_window_for_submission_row(row, final_test_df, framerate=60):
    # Filter by Filename and Time
    group = final_test_df[
        (final_test_df['Filename'] == row['Filename']) &
        (final_test_df['Time'] == row['Time'])
    ]
    group = group.sort_values(['minute', 'second', 'millisecond'])
    # Get the window (sequence) for the model
    window = group.iloc[:framerate]
    return window

# Build your test set in the same order as submission
test_windows = []
for idx, row in test_df.iterrows():
    window = get_window_for_submission_row(row, final_test_df, framerate=30)
    # ... process window as needed for your model ...
    test_windows.append(window)

In [ ]:
final_test_df = pd.concat(test_windows, ignore_index=True)

In [ ]:
final_test_df

In [ ]:
submission = pd.read_csv('/project/ai901504-ai0004/kaggle_competition/week7/test_submission.csv')
submission

In [ ]:
# Get unique pairs from both DataFrames
submission_pairs = set(zip(submission['Filename'], submission['Time']))
final_pairs = set(zip(final_test_df['Filename'], final_test_df['Time']))

# Check if they are the same
if submission_pairs == final_pairs:
    print("Unique pairs match between submission and final_test_df.")
else:
    print("Unique pairs do NOT match.")
    print("Pairs in submission but not in final_test_df:", submission_pairs - final_pairs)
    print("Pairs in final_test_df but not in submission:", final_pairs - submission_pairs)

In [ ]:
# Group by ('Filename', 'Time') and count rows
pair_counts = final_test_df.groupby(['Filename', 'Time']).size()

# Find pairs that do not have 50 rows
not_50 = pair_counts[pair_counts != 50]

if not_50.empty:
    print("All unique pairs in final_test_df have 50 rows.")
else:
    print("Some pairs do not have 50 rows:")
    print(not_50)

### train test split

In [ ]:
unique_pairs = final_df['subject_timestamp_pair'].unique()
train_pairs, test_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42)
train_pairs_list = list(train_pairs)
test_pairs_list = list(test_pairs)

In [ ]:
train_df = final_df[final_df['subject_timestamp_pair'].isin(train_pairs_list)].copy()
eval_df = final_df[final_df['subject_timestamp_pair'].isin(test_pairs_list)].copy()
# train_df = train_df.drop(columns=['subject_timestamp_pair'])
# test_df = test_df.drop(columns=['subject_timestamp_pair'])
eval_df

In [ ]:
train_df

In [ ]:
final_df

### Augment

In [ ]:
feature_cols = ['pos_0_x', 'pos_0_y',
      'pos_1_x', 'pos_1_y', 'pos_2_x', 'pos_2_y', 'pos_3_x', 'pos_3_y',
      'pos_4_x', 'pos_4_y', 'pos_5_x', 'pos_5_y', 'pos_6_x', 'pos_6_y',
      'pos_7_x', 'pos_7_y', 'pos_8_x', 'pos_8_y', 'pos_9_x', 'pos_9_y',
      'pos_10_x', 'pos_10_y', 'pos_11_x', 'pos_11_y', 'pos_12_x', 'pos_12_y']

# สำเนาหลักที่จะเก็บทั้งหมด
augment_df = train_df.copy()

# loop ทำ augmentation 4 แบบ
for i in range(4):
    copy_df = train_df.copy()
    
    for col in feature_cols:
        noise = np.random.uniform(0.95, 1.05, size=len(train_df))
        
        # ปรับตามเงื่อนไข
        if i == 0:
            copy_df[col] = train_df[col] + 100 * noise  # +x +y
        elif i == 1:
            if 'x' in col:
                copy_df[col] = train_df[col] + 100 * noise  # +x
            else:
                copy_df[col] = train_df[col] - 100 * noise  # -y
        elif i == 2:
            copy_df[col] = train_df[col] - 100 * noise  # -x -y
        elif i == 3:
            if 'x' in col:
                copy_df[col] = train_df[col] - 100 * noise  # -x
            else:
                copy_df[col] = train_df[col] + 100 * noise  # +y

    # เปลี่ยนชื่อเพื่อระบุชนิดการแปลง
    if i == 0:
        suffix = '_plusx_plusy'
    elif i == 1:
        suffix = '_plusx_minusy'
    elif i == 2:
        suffix = '_minusx_minusy'
    else:
        suffix = '_minusx_plusy'
    
    copy_df['Filename'] = copy_df['Filename'] + suffix
    copy_df['subject_timestamp_pair'] = copy_df['subject_timestamp_pair'].apply(
        lambda x: (x[0] + suffix, x[1])
    )
    
    # เพิ่มเข้า augment_df
    augment_df = pd.concat([augment_df, copy_df], ignore_index=True)

### Normalize

In [ ]:
f_df = final_df[feature_cols].replace(0, np.nan)
print(f_df.isnull().sum())

In [ ]:
scaler = StandardScaler()

# final_df[feature_cols] = final_df[feature_cols].fillna(0)
final_df[feature_cols] = final_df[feature_cols].replace(0, np.nan)
print(final_df.isnull().sum())
final_df[feature_cols] = final_df[feature_cols].fillna(method='ffill')
scaler.fit(final_df[feature_cols])

def scale_features(df, feature_cols, scaler):

   df_processed = df.copy()

   # Save NaN masks
   # nan_mask = df_processed[feature_cols].isnull()

   # Temporarily fill NaNs with 0s for scaler to work
   # df_processed[feature_cols] = df_processed[feature_cols].fillna(0)

   df_processed[feature_cols] = df_processed[feature_cols].fillna(method='ffill')

   # Fit on train and transform both
   df_processed[feature_cols] = scaler.transform(df_processed[feature_cols])

   # Restore missing values as sentinel (-99999)
   # df_processed[feature_cols] = df_processed[feature_cols].mask(nan_mask, -99999)
   
   return df_processed

In [ ]:
train_df = scale_features(train_df, feature_cols, scaler)
eval_df = scale_features(eval_df, feature_cols, scaler)
test_df = scale_features(final_test_df, feature_cols, scaler)
augment_df = scale_features(augment_df, feature_cols, scaler)

In [ ]:
eval_df

In [ ]:
train_df.to_parquet("/project/ai901504-ai0004/501641_Big/week7/dataset/train_smiley_no_augment.parquet")
# augment_df.to_parquet("/project/ai901504-ai0004/501641_Big/week7/dataset/train_smiley_with_augment.parquet")
eval_df.to_parquet('/project/ai901504-ai0004/501641_Big/week7/dataset/eval_smiley.parquet')

# Model

In [ ]:
class STGNNDataset(Dataset):
    def __init__(self, df, framerate=60):
        self.framerate = framerate
        self.samples = []

        self.label_columns = [
            'Advancing', 'Retreating', 'Enclosing', 'Spreading',
            'Rising', 'Descending', 'Directing', 'Indirecting',
            'Increasing Pressure', 'Decreasing Pressure', 'Accelerating',
            'Decelerating'
        ]

        # Group by subject and timestamp to ensure continuity
        grouped = df.groupby(['subject_timestamp_pair'])

        for _, group in grouped:
            group = group.sort_values(by=['minute', 'second', 'millisecond'])  # Ensure frame order

            # Only use full sequences
            if len(group) >= framerate:
                for start_idx in range(0, len(group) - framerate + 1):
                    window = group.iloc[start_idx:start_idx + framerate]
                    window_feature = window.drop(columns=[
                        'Filename', 'minute', 'second', 'millisecond', 'Time', 'subject_timestamp_pair'
                        ] + self.label_columns)

                    window_feature = window_feature.to_numpy()

                    X = window_feature.reshape(framerate, 13, 2)

                    # Assume same label for all frames in the sequence (use the first one)
                    y = window[self.label_columns].iloc[1].values.astype(np.float32)

                    self.samples.append((X, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]

        x = torch.tensor(x, dtype=torch.float32).permute(1, 2, 0)
        # x = torch.tensor(x, dtype=torch.float32).permute(2, 0, 1)

        y = torch.tensor(y, dtype=torch.float32)

        return x, y

In [ ]:
# train_ds = STGNNDataset(augment_df, framerate=50)
train_ds = STGNNDataset(train_df, framerate=50)
eval_ds = STGNNDataset(eval_df, framerate=50)

In [ ]:
print(train_ds[0])
print(train_ds[0][0].shape)
print(train_ds[0][1].shape)

### 1 experement

In [ ]:
from torch_geometric_temporal import STConv

class DeepSTGNN(nn.Module):
    def __init__(self, infea, outfea, L, d, num_nodes, kernel_size=3, K=3, framerate=30):
        super(DeepSTGNN, self).__init__()
        self.start_emb = nn.Linear(infea, outfea)
        
        # STConv layers for spatio-temporal feature extraction
        self.stconv_layers = nn.ModuleList([
            STConv(
                num_nodes=num_nodes,
                in_channels=outfea if i == 0 else outfea * 2,
                hidden_channels=outfea * 2,
                out_channels=outfea * 2,
                kernel_size=kernel_size,
                K=K
            ) for i in range(L)
        ])
        
        # Transform layers for attention-based processing
        self.transform = nn.ModuleList([Transform(outfea * 2, d) for _ in range(L)])
        self.positional_encoding = PositionalEncoding(outfea * 2, max_len=framerate)
        
        # Final layers
        self.global_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(outfea * 2, 12)
        
        self.L = L
        self.num_nodes = num_nodes

    def forward(self, x, edge_index, edge_weight=None):
        """
        Args:
            x: [B, T, N, infea] - Input tensor
            edge_index: [2, E] - Graph edge indices
            edge_weight: [E] - Edge weights (optional)
        Returns:
            output: [B, 12] - Classification logits
        """
        x = self.start_emb(x)  # [B, T, N, outfea]
        
        # Process through STConv layers
        for i in range(self.L):
            # STConv expects [B, F, N, T], so permute
            x_st = x.permute(0, 3, 1, 2)  # [B, F, N, T]
            x_st = self.stconv_layers[i](x_st, edge_index, edge_weight)
            x = x_st.permute(0, 3, 1, 2)  # [B, T, N, F]
        
        x = self.positional_encoding(x)
        for i in range(self.L):
            x = self.transform[i](x)
        
        x_pooled = self.global_pooling(x.permute(0, 3, 1, 2)).squeeze(-1).squeeze(-1)
        output = self.classifier(x_pooled)
        return output

In [ ]:
class Transform(nn.Module):
    def __init__(self, outfea, d):
        super(Transform, self).__init__()
        self.qff = nn.Linear(outfea, outfea)
        self.kff = nn.Linear(outfea, outfea)
        self.vff = nn.Linear(outfea, outfea)

        self.ln = nn.LayerNorm(outfea)
        self.lnff = nn.LayerNorm(outfea)

        self.ff = nn.Sequential(
            nn.Linear(outfea, outfea),
            nn.ReLU(),
            nn.Linear(outfea, outfea)
        )

        self.d = d

    def forward(self, x):
        B, T, N, F_out = x.shape
        num_heads = F_out // self.d

        if num_heads == 0:
            raise ValueError(f"Feature dimension ({F_out}) must be divisible by head dimension ({self.d})")

        query = self.qff(x)
        key = self.kff(x)
        value = self.vff(x)

        # Reshape for multi-head attention: (B, T, N, F_out) -> (B, T, N, num_heads, d) -> (B * num_heads, N, T, d)
        query = query.view(B, T, N, num_heads, self.d).permute(0, 3, 2, 1, 4).reshape(B * num_heads, N, T, self.d)
        key = key.view(B, T, N, num_heads, self.d).permute(0, 3, 2, 4, 1).reshape(B * num_heads, N, self.d, T) # Key is transposed last two dims
        value = value.view(B, T, N, num_heads, self.d).permute(0, 3, 2, 1, 4).reshape(B * num_heads, N, T, self.d)

        A = torch.matmul(query, key)
        A /= (self.d ** 0.5)
        A = torch.softmax(A, -1)

        value = torch.matmul(A ,value)
        value = value.view(B, num_heads, N, T, self.d).permute(0, 3, 2, 1, 4).reshape(B, T, N, F_out)
        value += x

        value = self.ln(value)
        x = self.ff(value) + value
        return self.lnff(x)


class PositionalEncoding(nn.Module):
    "Implement the PE function."
    def __init__(self, outfea, max_len=30):
        super(PositionalEncoding, self).__init__()

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, outfea).to(device)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, outfea, 2) *
                             -(math.log(10000.0) / outfea))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).unsqueeze(2) #[1,T,1,F]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + Variable(self.pe[:, :x.size(1), :, :], requires_grad=False)
        return x


class SGNN(nn.Module):
    def __init__(self, outfea):
        super(SGNN, self).__init__()
        self.ff = nn.Sequential(
            nn.Linear(outfea, outfea),
            nn.Linear(outfea, outfea)
        )
        self.ff1 = nn.Linear(outfea, outfea)

    def forward(self, x):
        p = self.ff(x)
        a = torch.matmul(p, p.transpose(-1,-2))
        R = torch.relu(torch.softmax(a, -1)) + torch.eye(x.shape[1]).to(device)

        D = (R.sum(-1) ** -0.5)
        D[torch.isinf(D)] = 0.
        D = torch.diag_embed(D)

        A = torch.matmul(torch.matmul(D, R), D)
        x = torch.relu(self.ff1(torch.matmul(A, x)))
        return x

class GRU(nn.Module):
    def __init__(self, outfea):
        super(GRU, self).__init__()
        self.ff = nn.Linear(2*outfea, 2*outfea)
        self.zff = nn.Linear(2*outfea, outfea)
        self.outfea = outfea

    def forward(self, x, xh):
        r, u = torch.split(torch.sigmoid(self.ff(torch.cat([x, xh], -1))), self.outfea, -1)
        z = torch.tanh(self.zff(torch.cat([x, r*xh], -1)))
        x = u * z + (1-u) * xh
        return x


class STGNNwithGRU(nn.Module):
    def __init__(self, outfea, framerate):
        super(STGNNwithGRU, self).__init__()
        self.sgnnh = nn.ModuleList([SGNN(outfea) for i in range(framerate)])
        self.sgnnx = nn.ModuleList([SGNN(outfea) for i in range(framerate)])
        self.gru = nn.ModuleList([GRU(outfea) for i in range(framerate)])
        self.framerate = framerate

    def forward(self, x):
        B,T,N,F = x.shape
        hidden_state = torch.zeros([B,N,F]).to(device)
        output = []

        for i in range(T):
            gx = self.sgnnx[i](x[:,i,:,:])
            gh = hidden_state
            if i != 0:
                gh = self.sgnnh[i](hidden_state)
            hidden_state = self.gru[i](gx, gh)
            output.append(hidden_state)

        output = torch.stack(output, 1)

        return output

class STGNN(nn.Module):
    def __init__(self, infea, outfea, L, d, framerate):
        super(STGNN, self).__init__()
        self.start_emb = nn.Linear(infea, outfea)

        self.stgnnwithgru = nn.ModuleList([STGNNwithGRU(outfea, framerate) for i in range(L)])
        self.positional_encoding = PositionalEncoding(outfea, max_len=framerate)
        self.transform = nn.ModuleList([Transform(outfea, d) for i in range(L)])

        self.L = L

        self.global_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(outfea, 12)

    def forward(self, x):
        '''
        x:[B,T,N, infea]
        '''
        x = self.start_emb(x)
        for i in range(self.L):
            x = self.stgnnwithgru[i](x)
        x = self.positional_encoding(x)
        for i in range(self.L):
            x = self.transform[i](x)
        x_pooled = self.global_pooling(x.permute(0, 3, 1, 2)).squeeze(-1).squeeze(-1)
        output = self.classifier(x_pooled)

        return output

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score

class STGNNTrainer:
    def __init__(self, model, edge_index, device='cuda', learning_rate=0.001, weight_decay=1e-4):
        self.model = model.to(device)
        self.edge_index = edge_index.to(device)  # Store edge_index on the device
        self.device = device
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', patience=10, factor=0.5)
        self.criterion = nn.BCEWithLogitsLoss()
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'learning_rate': []
        }
        self.best_val_loss = float('inf')
        self.best_model_state = None

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        all_predictions = []
        all_targets = []

        train_bar = tqdm(train_loader, desc='Training')
        for batch_idx, (data, target) in enumerate(train_bar):
            data, target = data.to(self.device), target.to(self.device)
            self.optimizer.zero_grad()
            output = self.model(data, self.edge_index)  # Pass edge_index
            loss = self.criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss += loss.item()
            predictions = torch.sigmoid(output) > 0.5
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.cpu().numpy())

            train_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Avg Loss': f'{total_loss/(batch_idx+1):.4f}'
            })

        all_predictions = np.vstack(all_predictions)
        all_targets = np.vstack(all_targets)
        accuracy = accuracy_score(all_targets, all_predictions)
        avg_loss = total_loss / len(train_loader)

        return avg_loss, accuracy, all_predictions, all_targets

    def evaluate(self, val_loader):
        self.model.eval()
        total_loss = 0
        all_predictions = []
        all_targets = []
        all_probabilities = []

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc='Validation')
            for data, target in val_bar:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data, self.edge_index)  # Pass edge_index
                loss = self.criterion(output, target)
                total_loss += loss.item()

                probabilities = torch.sigmoid(output)
                predictions = probabilities > 0.5
                all_predictions.append(predictions.cpu().numpy())
                all_targets.append(target.cpu().numpy())
                all_probabilities.append(probabilities.cpu().numpy())

                val_bar.set_postfix({'Loss': f'{loss.item():.4f}'})

        all_predictions = np.vstack(all_predictions)
        all_targets = np.vstack(all_targets)
        all_probabilities = np.vstack(all_probabilities)
        avg_loss = total_loss / len(val_loader)
        accuracy = accuracy_score(all_targets, all_predictions)

        return avg_loss, accuracy, all_predictions, all_targets, all_probabilities

    def train(self, train_loader, val_loader, epochs, save_path=None, early_stopping_patience=None):
        patience_counter = 0
        for epoch in range(epochs):
            print(f'Epoch {epoch+1}/{epochs}')
            train_loss, train_acc, _, _ = self.train_epoch(train_loader)
            val_metrics = self.evaluate(val_loader)
            val_loss, val_acc = val_metrics[0], val_metrics[1]

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_accuracy'].append(train_acc)
            self.history['val_accuracy'].append(val_acc)
            self.history['learning_rate'].append(self.optimizer.param_groups[0]['lr'])

            self.scheduler.step(val_loss)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_model_state = self.model.state_dict()
                if save_path:
                    torch.save(self.best_model_state, save_path)
                patience_counter = 0
            else:
                patience_counter += 1

            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
            print(f'Learning Rate: {self.optimizer.param_groups[0]["lr"]:.6f}')
            print('-' * 50)

            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f'Early stopping triggered after {epoch+1} epochs')
                break

    def predict(self, data_loader, threshold=0.5, return_probabilities=False):
        self.model.eval()
        all_predictions = []
        all_probabilities = []

        with torch.no_grad():
            for data, _ in tqdm(data_loader, desc='Predicting'):
                data = data.to(self.device)
                output = self.model(data, self.edge_index)  # Pass edge_index
                probabilities = torch.sigmoid(output)
                predictions = probabilities > threshold
                all_predictions.append(predictions.cpu().numpy())
                all_probabilities.append(probabilities.cpu().numpy())

        predictions = np.vstack(all_predictions)
        probabilities = np.vstack(all_probabilities)

        if return_probabilities:
            return predictions, probabilities
        return predictions

    def predict_single(self, x, threshold=0.5, return_probabilities=False):
        self.model.eval()
        if x.dim() == 3:
            x = x.unsqueeze(0)
        x = x.to(self.device)
        with torch.no_grad():
            output = self.model(x, self.edge_index)  # Pass edge_index
            probability = torch.sigmoid(output).cpu().numpy().squeeze()
            prediction = probability > threshold
        if return_probabilities:
            return prediction, probability
        return prediction

    # Add plot_training_history and other methods as needed...

In [ ]:
model = STGNN(infea=2, outfea=64, L=4, d=2, framerate=60)

trainer = STGNNTrainer(model, edge_index, device=device, learning_rate=0.001)

# Train model
trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    save_path='best_stgnn_model.pth',
    early_stopping_patience=15
)

# Plot training history
trainer.plot_training_history(save_path='training_history.png')

# Make predictions
predictions = trainer.predict(val_loader)
predictions_with_probs, probabilities = trainer.predict(val_loader, return_probabilities=True)

# Single sample prediction
single_sample = torch.randn(30, 17, 3)  # Example input
pred = trainer.predict_single(single_sample)
pred_with_prob, prob = trainer.predict_single(single_sample, return_probabilities=True)

# Evaluate model
val_metrics = trainer.evaluate(val_loader)
print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
print(f"Average F1 Score: {np.mean(val_metrics['f1']):.4f}")
print(f"Average AUC Score: {np.mean(val_metrics['auc']):.4f}")

# Plot confusion matrices
class_names = [f'Action_{i}' for i in range(12)]  # Replace with your actual class names
trainer.plot_confusion_matrix(val_loader, class_names=class_names, save_path='confusion_matrices.png')

### Jeang ded

In [ ]:
# Usage Example
def create_data_loaders(train_ds, eval_ds, batch_size=32, num_workers=16):
    """Create train and validation data loaders"""
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        eval_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader

In [ ]:
train_loader, val_loader = create_data_loaders(train_ds, eval_ds, batch_size=32)

In [ ]:
for batch in train_loader:
    print(batch[0].shape)
    break

In [ ]:
num_nodes = 13
# edge_index = torch.combinations(torch.arange(num_nodes), r=2).T  # [2, N]
# edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)  # Add reverse edges
# edge_index

edge_index = torch.tensor([[0, 0, 0, 0, 1, 1, 2, 3, 4, 5, 5, 5, 5,  6,  6,  6, 7,  8, 11, 1, 2, 3, 4, 2, 3, 4, 5, 6, 6, 7, 9, 11, 8, 10, 12, 9, 10, 12],
                           [1, 2, 3, 4, 2, 3, 4, 5, 6, 6, 7, 9, 11, 8, 10, 12, 9, 10, 12, 0, 0, 0, 0, 1, 1, 2, 3, 4, 5, 5, 5, 5,  6,  6,  6, 7,  8, 11]])

edge_index = edge_index.to(device)

In [ ]:
from torch_geometric_temporal.nn.attention import ASTGCN

model = ASTGCN(
    nb_block=4,
    in_channels=2,        # Your feature dimension per node (2)
    K=3,
    nb_chev_filter=128,
    nb_time_filter=128,
    time_strides=1,
    num_for_predict=12, # The output dimension (12)
    len_input=50,  
    num_of_vertices=13, # Your number of nodes (13)
    normalization="sym",
    bias=True
).to(device)



In [ ]:
import torch
import torch.nn as nn
from torch_geometric_temporal.nn.attention import AAGCN

class AAGCNClassifier(nn.Module):
    def __init__(self, in_channels, num_nodes, edge_index, num_classes, stride=1, residual=True, adaptive=True, attention=True):
        super().__init__()

        self.num_nodes = num_nodes
        self.stride = stride
        self.out_channels = 256  # จำนวน channel ที่จะใช้ภายใน AAGCN

        self.aagcn = AAGCN(
            in_channels=in_channels,
            out_channels=self.out_channels,
            edge_index=edge_index,
            num_nodes=num_nodes,
            stride=stride,
            residual=residual,
            adaptive=adaptive,
            attention=attention
        )

        # AAGCN output shape: (B, 256, T_out, num_nodes)
        # เราจะ average over nodes แล้ว flatten time
        T_out = 50 // stride  # Assuming input sequence length is 60
        self.final_classifier = nn.Linear(self.out_channels * T_out, num_classes)

    def forward(self, x):
        """
        x: Tensor with shape (B, F_in, T_in, N_nodes)
        """
        x = self.aagcn(x)  # → (B, 256, T_out, N_nodes)
        x = x.mean(dim=-1)  # mean over nodes → (B, 256, T_out)
        x = x.view(x.size(0), -1)  # flatten temporal dim → (B, 256 * T_out)
        out = self.final_classifier(x)  # → (B, num_classes)
        return out

model = AAGCNClassifier(
    in_channels=2,       # input feature per node
    num_nodes=13,
    edge_index=edge_index,
    num_classes=12,      # output class
    stride=1,
    residual=True,
    adaptive=True,
    attention=True
).to(device)

In [ ]:
from torch_geometric.nn import global_mean_pool


out = model(batch[0].to(device), edge_index)  # [1, 13, 12]
out = out.squeeze(0)                   # → [13, 12]
batch = torch.zeros(13, dtype=torch.long).to(device)

out = global_mean_pool(out, batch)     # → [1, 12]
print(out.shape)  

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import global_mean_pool
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ✅ Loss function
criterion = nn.BCEWithLogitsLoss()

# ✅ Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)


In [ ]:
num_epoch = 50

for epoch in range(num_epoch):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for x, y in tqdm(train_loader, desc="Training"):
        x = x.to(device)            # [B, 13, 2, 60]
        y = y.to(device)            # [B, 12]

        optimizer.zero_grad()

        out = model(x, edge_index)  # [B, 13, 12]

        batch = torch.arange(x.size(0), device=device).repeat_interleave(num_nodes)

        out = global_mean_pool(out.contiguous().view(-1, out.size(-1)), batch)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(out) > 0.5).float()

        all_preds.append(preds.detach().cpu())
        all_targets.append(y.detach().cpu())

    avg_loss = total_loss / len(train_loader)
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    all_preds_np = all_preds.cpu().tolist()
    all_targets_np = all_targets.cpu().tolist()

    acc = accuracy_score(all_targets_np, all_preds_np)
    f1 = f1_score(all_targets_np, all_preds_np, average='macro')

    model.eval()
    total_val_loss = 0
    all_val_preds = []
    all_val_targets = []
    with torch.no_grad():
        for x_val, y_val in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epoch} Validation"):
            x_val = x_val.to(device)
            y_val = y_val.to(device)

            # Forward pass
            out_val = model(x_val, edge_index)

            # Create batch vector for validation data
            batch_val = torch.arange(x_val.size(0), device=device).repeat_interleave(num_nodes)

            # Apply global_mean_pool
            out_val_pooled = global_mean_pool(out_val.reshape(-1, out_val.size(-1)), batch_val)

            # Calculate loss
            val_loss = criterion(out_val_pooled, y_val) # out_val_pooled
            total_val_loss += val_loss.item()

            # Get predictions
            preds_val = (torch.sigmoid(out_val_pooled) > 0.5).float() # out_val_pooled

            all_val_preds.append(preds_val.detach().cpu())
            all_val_targets.append(y_val.detach().cpu())

    # Calculate average validation loss and accuracy
    avg_val_loss = total_val_loss / len(val_loader)
    all_val_preds = torch.cat(all_val_preds, dim=0)
    all_val_targets = torch.cat(all_val_targets, dim=0)
    val_acc = accuracy_score(all_val_targets.cpu().tolist(), all_val_preds.cpu().tolist())
    val_f1 = f1_score(all_val_targets.cpu().tolist(), all_val_preds.cpu().tolist(), average='macro')


    print(f"Epoch {epoch+1}/{num_epoch}: "
        f"Train Loss: {avg_loss:.4f}, Train Acc: {acc:.4f}, Train F1: {f1:.4f} | "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

print("Training and Validation complete!")

In [ ]:
class STGNNDataset(Dataset):
    def __init__(self, df, framerate=60, has_labels=True):
        self.framerate = framerate
        self.samples = []
        self.has_labels = has_labels

        self.label_columns = [
            'Advancing', 'Retreating', 'Enclosing', 'Spreading',
            'Rising', 'Descending', 'Directing', 'Indirecting',
            'Increasing Pressure', 'Decreasing Pressure', 'Accelerating',
            'Decelerating'
        ]

        grouped = df.groupby(['subject_timestamp_pair'])

        for _, group in grouped:
            group = group.sort_values(by=['minute', 'second', 'millisecond'])

            if len(group) >= framerate:
                for start_idx in range(0, len(group) - framerate + 1):
                    window = group.iloc[start_idx:start_idx + framerate]
                    window_feature = window.drop(columns=[
                        'Filename', 'minute', 'second', 'millisecond', 'Time', 'subject_timestamp_pair'
                    ] + (self.label_columns if has_labels else []))

                    window_feature = window_feature.to_numpy()
                    X = window_feature.reshape(framerate, 13, 2)

                    if has_labels:
                        y = window[self.label_columns].iloc[1].values.astype(np.float32)
                        self.samples.append((X, y))
                    else:
                        self.samples.append(X)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.has_labels:
            x, y = self.samples[idx]
            x = torch.tensor(x, dtype=torch.float32).permute(1, 2, 0)
            y = torch.tensor(y, dtype=torch.float32)
            return x, y
        else:
            x = self.samples[idx]
            x = torch.tensor(x, dtype=torch.float32).permute(1, 2, 0)
            return x

In [ ]:
test_dataset = STGNNDataset(test_df, framerate=50, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model.eval()
all_preds = []
with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        out = model(x, edge_index)
        batch = torch.arange(x.size(0), device=device).repeat_interleave(num_nodes)
        out_pooled = global_mean_pool(out.reshape(-1, out.size(-1)), batch)
        preds = (torch.sigmoid(out_pooled) > 0.5).float()
        all_preds.append(preds.cpu())

all_preds = torch.cat(all_preds, dim=0)

In [ ]:
all_preds.shape

In [ ]:
submission = pd.read_csv('/project/ai901504-ai0004/kaggle_competition/week7/test_submission.csv')

In [ ]:
submission

In [ ]:
all_preds_list = all_preds.tolist()

label_columns = [
    'Advancing', 'Retreating', 'Enclosing', 'Spreading',
    'Rising', 'Descending', 'Directing', 'Indirecting',
    'Increasing Pressure', 'Decreasing Pressure', 'Accelerating',
    'Decelerating'
]

df_preds = pd.DataFrame(all_preds_list, columns=label_columns)

In [ ]:
df_preds

In [ ]:
label_columns = [
    'Advancing', 'Retreating', 'Enclosing', 'Spreading',
    'Rising', 'Descending', 'Directing', 'Indirecting',
    'Increasing Pressure', 'Decreasing Pressure', 'Accelerating', 'Decelerating'
]

# Replace columns by position (row order must match!)
submission[label_columns] = df_preds[label_columns].values

In [ ]:
submission.isnull().sum()

In [ ]:
submission = submission.fillna(0)

In [ ]:
submission.to_parquet('/project/ai901504-ai0004/501641_Big/week7/submission.parquet')